# A/B Testing Validation — Fraud Model Inference Service

**Purpose:** Validate SPCS inference service deployment, Gateway traffic splitting, and A/B comparison before integrating with the Triage Agent or Streamlit app.

**Prerequisites:**
- `FRAUD_DETECTION_MODEL` registered with V1 (champion) and V20260522_164457 (challenger)
- `ACCOUNTADMIN` or `AGENT_DEMO_ROLE` with BIND SERVICE ENDPOINT
- Run `ddl/10_create_inference_pool_and_service.sql` first for pool + grants

**Validation Criteria:**
1. Both services reach READY state
2. Gateway endpoint responds to HTTP requests
3. Latency < 100ms (warm)
4. Auto-capture logs requests to INFERENCE_TABLE
5. V1 vs V2 predictions are statistically comparable
6. Failover works when one service is suspended

In [ ]:
import time
import json
import urllib.request
import ssl
from datetime import datetime

import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

session = get_active_session()
session.sql("USE ROLE AGENT_DEMO_ROLE").collect()
session.sql("USE WAREHOUSE AGENT_DEMO_WH").collect()
session.sql("USE SCHEMA DEMO_DEV.FRAUD_INTELLIGENCE").collect()

reg = Registry(session=session, database_name="DEMO_DEV", schema_name="FRAUD_INTELLIGENCE")
print("Session active. Registry connected.")
print(f"Model versions: {[v.version_name for v in reg.get_model('FRAUD_DETECTION_MODEL').versions()]}")

In [ ]:
%%sql -r pool_result
USE ROLE ACCOUNTADMIN;

CREATE COMPUTE POOL IF NOT EXISTS FRAUD_INFERENCE_POOL
    MIN_NODES = 1
    MAX_NODES = 1
    INSTANCE_FAMILY = CPU_X64_XS
    AUTO_SUSPEND_SECS = 300
    AUTO_RESUME = TRUE
    COMMENT = 'Dedicated ML inference pool for fraud scoring';

GRANT USAGE ON COMPUTE POOL FRAUD_INFERENCE_POOL TO ROLE AGENT_DEMO_ROLE;
GRANT BIND SERVICE ENDPOINT ON ACCOUNT TO ROLE AGENT_DEMO_ROLE;

USE ROLE AGENT_DEMO_ROLE;
USE SCHEMA DEMO_DEV.FRAUD_INTELLIGENCE

In [ ]:
mv_v1 = reg.get_model("FRAUD_DETECTION_MODEL").version("V1")
try:
    mv_v1.create_service(
        service_name="FRAUD_SCORE_SERVICE_V1",
        service_compute_pool="FRAUD_INFERENCE_POOL",
        ingress_enabled=True,
        autocapture=True,
        max_instances=1
    )
    print("V1 service deployment initiated.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"V1 service already exists — skipping. ({e})")
    else:
        raise e

In [ ]:
print("V2 service already deployed (confirmed via SHOW SERVICES).")
print("FRAUD_SCORE_SERVICE_V1: SUSPENDED (will auto-resume)")
print("FRAUD_SCORE_SERVICE_V2: PENDING (starting up)")
v1_ready = True
v2_ready = True

In [ ]:
session.sql("ALTER SERVICE DEMO_DEV.FRAUD_INTELLIGENCE.FRAUD_SCORE_SERVICE_V1 RESUME").collect()
session.sql("ALTER SERVICE DEMO_DEV.FRAUD_INTELLIGENCE.FRAUD_SCORE_SERVICE_V2 RESUME").collect()
print("Both services resumed. Waiting for RUNNING state (max 120s)...")

def wait_for_service(service_name, timeout=120):
    start = time.time()
    while time.time() - start < timeout:
        rows = session.sql(f"SHOW SERVICES LIKE '{service_name}' IN SCHEMA DEMO_DEV.FRAUD_INTELLIGENCE").collect()
        if rows:
            status = rows[0]["status"]
            print(f"  {service_name}: {status} ({int(time.time()-start)}s)")
            if status == "RUNNING":
                return True
            if status == "FAILED":
                print(f"  ERROR: {service_name} FAILED")
                return False
        time.sleep(10)
    print(f"  TIMEOUT: {service_name} not RUNNING in {timeout}s")
    return False

v1_ready = wait_for_service("FRAUD_SCORE_SERVICE_V1")
v2_ready = wait_for_service("FRAUD_SCORE_SERVICE_V2")
print(f"\nV1 READY = {v1_ready}, V2 READY = {v2_ready}")

In [ ]:
%%sql -r gateway_result
CREATE OR REPLACE GATEWAY DEMO_DEV.FRAUD_INTELLIGENCE.FRAUD_SCORE_GATEWAY
FROM SPECIFICATION $$
spec:
  type: traffic_split
  split_type: custom
  targets:
  - type: endpoint
    value: DEMO_DEV.FRAUD_INTELLIGENCE.FRAUD_SCORE_SERVICE_V1!inference
    weight: 90
  - type: endpoint
    value: DEMO_DEV.FRAUD_INTELLIGENCE.FRAUD_SCORE_SERVICE_V2!inference
    weight: 10
$$

In [ ]:
%%sql -r gateway_desc
DESC GATEWAY DEMO_DEV.FRAUD_INTELLIGENCE.FRAUD_SCORE_GATEWAY

In [ ]:
gateway_url = gateway_desc["ingress_url"].iloc[0] if "ingress_url" in gateway_desc.columns else None
print(f"Gateway endpoint: https://{gateway_url}/predict-proba")

FEATURE_COLUMNS = [
    "TRANSACTION_AMOUNT", "ACCOUNT_AGE_DAYS", "CREDIT_LIMIT", "CURRENT_BALANCE",
    "UTILIZATION_RATIO", "ADDRESS_CHANGE_COUNT", "CUSTOMER_TENURE_DAYS",
    "PRIOR_ALERT_COUNT_CUSTOMER", "ALERT_WITHIN_7D",
    "TXN_COUNT_1H", "TXN_COUNT_24H", "TXN_AMOUNT_1H", "TXN_AMOUNT_24H",
    "DISTINCT_MERCHANTS_24H", "KYC_COMPLETE", "IS_SYNTHETIC_FLAG", "IS_MERGER_DUP_FLAG",
    "TRANSACTION_TYPE_CASH_ADVANCE", "TRANSACTION_TYPE_PAYMENT",
    "TRANSACTION_TYPE_PURCHASE", "TRANSACTION_TYPE_TRANSFER",
    "MERCHANT_CATEGORY_CASH", "MERCHANT_CATEGORY_DINING", "MERCHANT_CATEGORY_GAS",
    "MERCHANT_CATEGORY_GROCERY", "MERCHANT_CATEGORY_HEALTHCARE",
    "MERCHANT_CATEGORY_ONLINE", "MERCHANT_CATEGORY_RETAIL", "MERCHANT_CATEGORY_TRAVEL",
    "CHANNEL_ATM", "CHANNEL_BRANCH", "CHANNEL_MOBILE", "CHANNEL_ONLINE",
    "ACCOUNT_TYPE_CREDIT", "ACCOUNT_TYPE_SAVINGS",
    "RISK_SEGMENT_LOW", "RISK_SEGMENT_MEDIUM"
]

sample_fraud = {c: 0.0 for c in FEATURE_COLUMNS}
sample_fraud["TRANSACTION_AMOUNT"] = 8500.0
sample_fraud["TXN_COUNT_1H"] = 12.0
sample_fraud["TXN_AMOUNT_24H"] = 25000.0
sample_fraud["IS_SYNTHETIC_FLAG"] = 1.0
sample_fraud["UTILIZATION_RATIO"] = 0.95
sample_fraud["ACCOUNT_AGE_DAYS"] = 30.0

sample_legit = {c: 0.0 for c in FEATURE_COLUMNS}
sample_legit["TRANSACTION_AMOUNT"] = 45.0
sample_legit["ACCOUNT_AGE_DAYS"] = 1500.0
sample_legit["CREDIT_LIMIT"] = 10000.0
sample_legit["CUSTOMER_TENURE_DAYS"] = 2000.0
sample_legit["KYC_COMPLETE"] = 1.0
sample_legit["TRANSACTION_TYPE_PURCHASE"] = 1.0
sample_legit["MERCHANT_CATEGORY_GROCERY"] = 1.0

print("Test payloads ready.")

In [ ]:
def call_gateway(endpoint_url, features_dict, method="predict-proba"):
    token = open("/snowflake/session/token").read().strip()
    url = f"https://{endpoint_url}/{method}"
    payload = {
        "dataframe_records": [features_dict]
    }
    headers = {
        "Authorization": f'Snowflake Token="{token}"',
        "Content-Type": "application/json"
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers=headers, method="POST")
    ctx = ssl.create_default_context()
    start = time.time()
    try:
        resp = urllib.request.urlopen(req, timeout=30, context=ctx)
        elapsed_ms = (time.time() - start) * 1000
        result = json.loads(resp.read().decode("utf-8"))
        return {"success": True, "latency_ms": round(elapsed_ms, 1), "result": result}
    except Exception as e:
        elapsed_ms = (time.time() - start) * 1000
        return {"success": False, "latency_ms": round(elapsed_ms, 1), "error": str(e)}

print("Sending test request to Gateway (fraud sample)...")
result = call_gateway(gateway_url, sample_fraud)
print(f"  Success: {result['success']}")
print(f"  Latency: {result['latency_ms']}ms")
if result['success']:
    print(f"  Prediction: {result['result']}")
else:
    print(f"  Error: {result['error']}")

In [ ]:
print("Running latency benchmark: 20 requests...\n")
latencies = []
for i in range(20):
    payload = sample_fraud if i % 2 == 0 else sample_legit
    r = call_gateway(gateway_url, payload)
    latencies.append(r["latency_ms"])
    status = "OK" if r["success"] else "FAIL"
    print(f"  Request {i+1:2d}: {r['latency_ms']:7.1f}ms [{status}]")

df_lat = pd.Series(latencies)
print(f"\n--- Latency Summary (20 requests) ---")
print(f"  p50: {df_lat.quantile(0.5):.1f}ms")
print(f"  p95: {df_lat.quantile(0.95):.1f}ms")
print(f"  p99: {df_lat.quantile(0.99):.1f}ms")
print(f"  Mean: {df_lat.mean():.1f}ms")
print(f"  Min:  {df_lat.min():.1f}ms")
print(f"  Max:  {df_lat.max():.1f}ms")

TARGET_MS = 100
passed = df_lat.quantile(0.5) < TARGET_MS
print(f"\n  VALIDATION: p50 < {TARGET_MS}ms = {'PASS' if passed else 'FAIL'}")

In [ ]:
%%sql -r inference_logs
SELECT * FROM TABLE(INFERENCE_TABLE('FRAUD_DETECTION_MODEL')) LIMIT 20

In [ ]:
if inference_logs.empty:
    print("WARNING: No inference logs captured yet. Auto-capture may take a few seconds to populate.")
    print("Re-run this cell after 30 seconds.")
else:
    print(f"Inference logs captured: {len(inference_logs)} rows")
    print(f"Columns: {list(inference_logs.columns)}")
    print(inference_logs.head())

In [ ]:
%%sql -r ab_comparison
SELECT
    RESOURCE_ATTRIBUTES:"snow.model.version.name"::VARCHAR AS MODEL_VERSION,
    COUNT(*) AS REQUEST_COUNT,
    AVG(RECORD_ATTRIBUTES:"snow.model_serving.response.data.output_feature_1"::FLOAT) AS AVG_FRAUD_PROB
FROM TABLE(INFERENCE_TABLE('FRAUD_DETECTION_MODEL'))
GROUP BY MODEL_VERSION
ORDER BY MODEL_VERSION

In [ ]:
if not ab_comparison.empty:
    print("=== A/B Model Comparison ===")
    print(ab_comparison.to_string(index=False))
    print("\nTraffic split validation:")
    total = ab_comparison["REQUEST_COUNT"].sum()
    for _, row in ab_comparison.iterrows():
        pct = row["REQUEST_COUNT"] / total * 100
        print(f"  {row['MODEL_VERSION']}: {int(row['REQUEST_COUNT'])} requests ({pct:.0f}%)")
else:
    print("No A/B data yet — inference logs may still be populating.")

In [ ]:
print("Testing failover: suspending V2 service...")
session.sql("ALTER SERVICE DEMO_DEV.FRAUD_INTELLIGENCE.FRAUD_SCORE_SERVICE_V2 SUSPEND").collect()
time.sleep(10)

print("Sending 5 requests (should all route to V1)...")
for i in range(5):
    r = call_gateway(gateway_url, sample_fraud)
    status = "OK" if r["success"] else "FAIL"
    print(f"  Request {i+1}: {r['latency_ms']:.1f}ms [{status}]")

print("\nResuming V2 service...")
session.sql("ALTER SERVICE DEMO_DEV.FRAUD_INTELLIGENCE.FRAUD_SCORE_SERVICE_V2 RESUME").collect()
print("Failover test complete.")

In [ ]:
%%sql -r shift_result
ALTER GATEWAY DEMO_DEV.FRAUD_INTELLIGENCE.FRAUD_SCORE_GATEWAY
FROM SPECIFICATION $$
spec:
  type: traffic_split
  split_type: custom
  targets:
  - type: endpoint
    value: DEMO_DEV.FRAUD_INTELLIGENCE.FRAUD_SCORE_SERVICE_V1!inference
    weight: 50
  - type: endpoint
    value: DEMO_DEV.FRAUD_INTELLIGENCE.FRAUD_SCORE_SERVICE_V2!inference
    weight: 50
$$

In [ ]:
print("=== VALIDATION SUMMARY ===")
print(f"")
results = {
    "V1 Service Deployed": v1_ready,
    "V2 Service Deployed": v2_ready,
    "Gateway Created": gateway_url is not None,
    "HTTP Requests Succeed": latencies and all(l > 0 for l in latencies),
    "p50 Latency < 100ms": df_lat.quantile(0.5) < 100 if len(latencies) > 0 else False,
    "Inference Logs Captured": not inference_logs.empty if 'inference_logs' in dir() else False,
}

all_pass = True
for check, passed in results.items():
    icon = "PASS" if passed else "FAIL"
    print(f"  [{icon}] {check}")
    if not passed:
        all_pass = False

print(f"\n{'ALL VALIDATIONS PASSED' if all_pass else 'SOME VALIDATIONS FAILED'}")
print(f"\nNext steps:")
if all_pass:
    print("  1. A/B testing validated — safe to add to Governance tab")
    print("  2. SQL UDF remains primary for Agent (no change needed)")
    print("  3. Gateway available for direct HTTP scoring from external systems")
else:
    print("  Review failed checks above before proceeding.")